# __Finantial Statemnet 2021__

## 1. __Data Loading__

In [26]:
from financial_project.azure.storage import get_filesystem

filesystem = get_filesystem('interim')

# List paths inside the filesystem
for path in filesystem.get_paths(): 
    print(path.name)

financial_statements
financial_statements/2021
financial_statements/2021/balances_2021_1.txt
financial_statements/2021/balances_2021_2.txt
financial_statements/2021/balances_2021_3.txt
financial_statements/2021/catalogo_2021_1.txt
financial_statements/2021/catalogo_2021_2.txt
financial_statements/2021/catalogo_2021_3.txt
financial_statements/2022
financial_statements/2022/balances_2022_1.txt
financial_statements/2022/balances_2022_2.txt
financial_statements/2022/balances_2022_3.txt
financial_statements/2022/catalogo_2022_1.txt
financial_statements/2022/catalogo_2022_2.txt
financial_statements/2022/catalogo_2022_3.txt
financial_statements/2023
financial_statements/2023/balances_2023_1.txt
financial_statements/2023/balances_2023_2.txt
financial_statements/2023/catalogo_2023_1.txt
financial_statements/2023/catalogo_2023_2.txt
financial_statements/2024
financial_statements/2024/balances_2024_1.txt
financial_statements/2024/catalogo_2024_1.txt
financial_statements/2025
financial_statements/

In [2]:
# Import the function that accesses, reads the Azure filesystem.
from financial_project.azure.readers import read_file_azure

# Balances 2021_1
data1 = read_file_azure(
    filesystem,
    'financial_statements/2025/balances_2025_1.txt'
)

# Balances 2021_2
data2 = read_file_azure(
    filesystem, 
    'financial_statements/2021/balances_2021_2.txt',
    encoding='utf-8'
)

# Balances 2021_3
data3 = read_file_azure(
    filesystem,
    'financial_statements/2021/balances_2021_3.txt',
    encoding='utf-8'
)

display(data1.head(2),data2.head(2),data3.head(2))

,AÑO,EXPEDIENTE,RUC,NOMBRE,RAMA_ACTIVIDAD,DESCRIPCION_RAMA,CIIU,CUENTA_1,CUENTA_101,CUENTA_10101,...,CUENTA_80004,CUENTA_80005,CUENTA_80006,CUENTA_80007,CUENTA_80008,CUENTA_80009,CUENTA_801,CUENTA_80101,CUENTA_80102,Unnamed: 629
0,2025,1,1790013731001,ACEITES TROPICALES SOCIEDAD ANONIMA ATSA,A,"AGRICULTURA, GANADERÍA, SILVICULTURA Y PESCA.",A0126.01,1.329499e+06,9.638320e+03,6338.46,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,2025,2,1790004724001,ACERIA DEL ECUADOR CA ADELCA.,C,INDUSTRIAS MANUFACTURERAS.,C2410.25,5.264843e+08,2.481542e+08,28574349.24,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


,A�O,EXPEDIENTE,RUC,NOMBRE,RAMA_ACTIVIDAD,DESCRIPCION_RAMA,CIIU,CUENTA_10,CUENTA_11,CUENTA_12,...,CUENTA_919,CUENTA_920,CUENTA_925,CUENTA_94,CUENTA_950,CUENTA_960,CUENTA_97,CUENTA_98,CUENTA_999,Unnamed: 875
0,2021,320915,1793194272001,ARTOPIN CONSULTOR�A & CONSTRUCCI�N S.A.S.,F,CONSTRUCCI�N.,F4100.10,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN


,A�O,EXPEDIENTE,RUC,NOMBRE,RAMA_ACTIVIDAD,DESCRIPCION_RAMA,CIIU,CUENTA_10,CUENTA_1025,CUENTA_1030,...,CUENTA_7903,CUENTA_7904,CUENTA_7905,CUENTA_7906,CUENTA_7907,CUENTA_7908,CUENTA_7991,CUENTA_7992,CUENTA_7999,Unnamed: 932
0,2021,1,1.790014e+12,ACEITES TROPICALES SOCIEDAD ANONIMA ATSA,A,"AGRICULTURA, GANADER�A, SILVICULTURA Y PESCA.",A0126.01,0.0,-144370.63,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,144370.63,32319.42,176690.05,NaN
1,2021,3,1.790009e+12,ACERO COMERCIAL ECUATORIANO S.A.,G,COMERCIO AL POR MAYOR Y AL POR MENOR REPARACI�...,G4610.03,0.0,-7820189.20,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,7820189.17,3745372.96,11565562.10,NaN


In [3]:
import pandas as pd

# Duplicate Values
values = [ "EXPEDIENTE", "RUC", "CIIU"]

for i in values : 

    duplicates = data1[i].isin(
    pd.concat([
        data2[i],
        data3[i]
    ])).sum()

    print(f"Variable: {i}\n Duplicates: {duplicates}\n")

Variable: EXPEDIENTE
 Duplicates: 77037

Variable: RUC
 Duplicates: 0

Variable: CIIU
 Duplicates: 149152



## 2. __Account Mapping & Filter__

__Financial Statement (IFRS)__

In [4]:
# Account Mapping
from financial_project.account_mapping import ifrs_accounts

# Filter
ifrs = data1[ifrs_accounts].copy()

__SRI Form 101__

In [8]:
# Account Mapping
from financial_project.account_mapping import sri_accounts

# Filter
data3 = data3.rename(columns={'A�O': 'AÑO'})
sri = data3[sri_accounts].copy()

## __3. Account Harmonization__

__Financial Statement (IFRS)__

In [9]:
# Accounts to be subtracted
ifrs["-CUENTA_501"] = -ifrs["CUENTA_501"]
ifrs["-CUENTA_502"] = -ifrs["CUENTA_502"]

__SRI Form 101__

In [10]:
# Accounts to be subtracted
sri["-CUENTA_347"] = -sri["CUENTA_347"]
sri["-CUENTA_7991"] = -sri["CUENTA_7991"]
sri["-CUENTA_1030"] = -sri["CUENTA_1030"]

## __4. Full Merge__

In [23]:
from financial_project.account_harmonization import sri_direct_mapping, sri_aggregated_mapping, ifrs_direct_mapping, ifrs_aggregated_mapping

def unificar_mapping(directa: dict, agrupada: dict) -> dict:
    mapping = {k: [v] for k, v in directa.items()}   # directa -> lista de 1 elemento
    mapping.update(agrupada)                          # agrupada ya viene como lista
    return mapping

ifrs_mapping = unificar_mapping(ifrs_direct_mapping, ifrs_aggregated_mapping)
sri_mapping  = unificar_mapping(sri_direct_mapping, sri_aggregated_mapping)

In [24]:
def build_harmonized_base(df: pd.DataFrame, mapping: dict, id_cols: list) -> pd.DataFrame:
    resultado = df[id_cols].copy()

    for nombre_estandar, columnas in mapping.items():
        resultado[nombre_estandar] = df[columnas].sum(axis=1)

    return resultado

In [25]:
data2021 = pd.DataFrame()

In [27]:
# Define the identification columns
ID_COLS = [
    "AÑO",
    "EXPEDIENTE",
    "RUC",
    "NOMBRE",
    "RAMA_ACTIVIDAD",
    "DESCRIPCION_RAMA",
    "CIIU",
]

# Build the harmonized datasets
sri_base = build_harmonized_base(sri, sri_mapping, ID_COLS)
ifrs_base = build_harmonized_base(ifrs, ifrs_mapping, ID_COLS)

# Add the source identifier
sri_base["ORIGEN"] = "SRI_101"
ifrs_base["ORIGEN"] = "NIIF"

# Combine both harmonized datasets
data2021 = pd.concat([ifrs_base, sri_base], ignore_index=True)

In [ ]:
data2021.to_parquet()

data2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 241470 entries, 0 to 241469
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   AÑO                        241470 non-null  int64  
 1   EXPEDIENTE                 241470 non-null  int64  
 2   RUC                        241469 non-null  object 
 3   NOMBRE                     241470 non-null  str    
 4   RAMA_ACTIVIDAD             241443 non-null  str    
 5   DESCRIPCION_RAMA           241443 non-null  str    
 6   CIIU                       241443 non-null  str    
 7   CURRENT ASSETS (CA)        241470 non-null  float64
 8   CURRENT LIABILITIES (CL)   241470 non-null  float64
 9   CASH AND CASH EQUIVALENTS  241470 non-null  float64
 10  TOTAL LIABILITIES          241470 non-null  float64
 11  EQUITY                     241470 non-null  float64
 12  TOTAL ASSETS               241470 non-null  float64
 13  NET INCOME                 241470 non-nu

In [31]:
# Connect to the processed filesystem.
filesystem = get_filesystem("processed")

# List the items at the root of the filesystem.
for item in filesystem.get_paths():
    print(
        f"Name: {item.name} | "
        f"Is directory: {item.is_directory}"
    )

Name: homologation | Is directory: True


In [ ]:
from io import BytesIO

filesystem = get_filesystem("processed")

# Convert the existing DataFrame to CSV bytes.
buffer = BytesIO()
data2021.to_csv(buffer, index=False, encoding="utf-8")
payload = buffer.getvalue()

# Define the destination file inside the existing directory.
file_path = "homologation/data2021.csv"
file_client = filesystem.get_file_client(file_path)

# Create the remote file before writing any data.
file_client.create_file()

# Append the CSV bytes starting at position zero.
file_client.append_data(
    data=payload,
    offset=0,
    length=len(payload),
)

# Commit the uploaded bytes so the file is finalized.
file_client.flush_data(len(payload))

# Confirm the upload.
print(f"Uploaded successfully: processed/{file_path}")

Uploaded successfully: processed/homologation/data2021.csv
